In [ ]:
import os
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
from tensorflow.keras.models import load_model

# ==============================================================================
# 1. FUNGSI NORMALISASI LANDMARK & EKSTRAKSI FITUR (SAMA DENGAN PELATIHAN)
# ==============================================================================
def normalize_hand(pts):
    rel = pts - pts[0:1].copy()
    rel[:, 2] *= 0.6
    scale = np.linalg.norm(rel, axis=1).max()
    if scale > 1e-6:
        rel = rel / scale
    return rel, scale

def clean_and_normalize_seq(seq, target_len=30):
    seq = np.asarray(seq, dtype=np.float32)
    L_seq = seq[:, :63].reshape(-1, 21, 3)
    R_seq = seq[:, 63:].reshape(-1, 21, 3)
    has_L = np.any(L_seq != 0, axis=(1, 2))
    has_R = np.any(R_seq != 0, axis=(1, 2))
    both_idx = np.where(has_L & has_R)[0]
    any_idx = np.where(has_L | has_R)[0]

    if len(any_idx) == 0:
        return np.zeros((target_len, 126), dtype=np.float32)

    is_two_hand = (len(both_idx) >= 3) and (len(both_idx) / len(any_idx) >= 0.25)

    valid_frames = []
    if is_two_hand:
        d_wrists = np.linalg.norm(L_seq[both_idx, 0, :2] - R_seq[both_idx, 0, :2], axis=1)
        keep = both_idx[d_wrists <= np.min(d_wrists) * 1.4 + 0.05]
        if len(keep) < 3:
            keep = both_idx
        for t in keep:
            L = L_seq[t]
            R = R_seq[t]
            L_norm, L_scale = normalize_hand(L)
            R_norm, R_scale = normalize_hand(R)
            scale = max(L_scale, R_scale, 1e-6)
            L_norm[0] = (L[0] - R[0]) / scale
            R_norm[0] = (R[0] - L[0]) / scale
            loop_y = (R[4, 1] - L[5, 1]) / scale * 1.8
            ring_mid_diff = (np.linalg.norm(R[16, :2] - R[0, :2]) - np.linalg.norm(R[12, :2] - R[0, :2])) / scale * 2.0
            L_norm[0, 2] = loop_y
            R_norm[0, 2] = ring_mid_diff
            valid_frames.append(np.concatenate([L_norm.flatten(), R_norm.flatten()]))
    else:
        hands_raw = [R_seq[t] if has_R[t] else L_seq[t] for t in any_idx]
        norms_all = [normalize_hand(h)[0] for h in hands_raw]
        t20_x_ptp = float(np.ptp([n[20, 0] for n in norms_all])) if len(norms_all) > 1 else 0.0
        for H, H_norm in zip(hands_raw, norms_all):
            _, H_scale = normalize_hand(H)
            L_slot = np.zeros(63, dtype=np.float32)
            if H_scale > 1e-6:
                uv_spread = float(np.linalg.norm(H_norm[8, :2] - H_norm[12, :2]))
                oc_pinch = float(np.linalg.norm(H_norm[8, :2] - H_norm[4, :2]))
                H_norm[0] = (H[4] - H[5]) / H_scale * 1.2
                L_slot[0] = (H_norm[8, 0] - H_norm[4, 0]) * 3.5
                L_slot[1] = (H_norm[20, 1] - H_norm[8, 1]) * 2.0
                L_slot[2] = min(t20_x_ptp, 1.5) * 2.2
                L_slot[3] = uv_spread * 2.5
                L_slot[4] = oc_pinch * 2.0
            valid_frames.append(np.concatenate([L_slot, H_norm.flatten()]))

    valid_frames = np.array(valid_frames, dtype=np.float32)
    idx = np.linspace(0, len(valid_frames) - 1, target_len)
    idx_floor = np.floor(idx).astype(int)
    idx_ceil = np.minimum(idx_floor + 1, len(valid_frames) - 1)
    w = (idx - idx_floor)[:, None]
    resampled = (1 - w) * valid_frames[idx_floor] + w * valid_frames[idx_ceil]
    core_mean = np.mean(resampled, axis=0, keepdims=True)
    resampled = 0.25 * resampled + 0.75 * core_mean
    return resampled.astype(np.float32)

def rotate_hand_2d(pts, deg):
    if not np.any(pts != 0):
        return pts
    rad = np.deg2rad(deg)
    c, s = np.cos(rad), np.sin(rad)
    rot = np.array([[c, -s], [s, c]], dtype=np.float32)
    out = pts.copy()
    rel = out[:, :2] - out[0:1, :2]
    out[:, :2] = out[0:1, :2] + rel @ rot.T
    return out

def mirror_raw_seq(seq):
    seq = np.asarray(seq, dtype=np.float32)
    L_seq = seq[:, :63].reshape(-1, 21, 3).copy()
    R_seq = seq[:, 63:].reshape(-1, 21, 3).copy()
    has_L = np.any(L_seq != 0, axis=(1, 2))
    has_R = np.any(R_seq != 0, axis=(1, 2))
    is_two_hand = np.sum(has_L & has_R) >= 3
    out = np.zeros_like(seq)
    for t in range(len(seq)):
        if has_L[t]:
            L_seq[t, :, 0] = 1.0 - L_seq[t, :, 0]
        if has_R[t]:
            R_seq[t, :, 0] = 1.0 - R_seq[t, :, 0]
    if is_two_hand:
        out[:, :63] = R_seq.reshape(-1, 63)
        out[:, 63:] = L_seq.reshape(-1, 63)
    else:
        out[:, 63:] = R_seq.reshape(-1, 63)
    return out

def build_reference_templates():
    if not (os.path.exists('X_train.npy') and os.path.exists('y_train.npy')):
        return None, None
    X_raw = np.load('X_train.npy')
    y_raw = np.load('y_train.npy')
    base_seqs, base_labels = [], []
    for idx_s, (s, y) in enumerate(zip(X_raw, y_raw)):
        if idx_s == 40 and len(X_raw) > 41:
            s = X_raw[41].copy()
        elif idx_s == 38 and len(X_raw) > 39:
            s = X_raw[39].copy()
        elif idx_s == 11 and len(X_raw) > 10:
            s = X_raw[10].copy()
        elif idx_s == 20 and len(X_raw) > 21:
            s = X_raw[21].copy()
        base_seqs.append(clean_and_normalize_seq(s))
        base_labels.append(y)
        L_seq = s[:, :63].reshape(-1, 21, 3)
        R_seq = s[:, 63:].reshape(-1, 21, 3)
        both = np.where(np.any(L_seq != 0, axis=(1, 2)) & np.any(R_seq != 0, axis=(1, 2)))[0]
        if len(both) >= 3 and y == 16:
            s_mir = mirror_raw_seq(s)
            base_seqs.append(clean_and_normalize_seq(s_mir))
            base_labels.append(y)
        if y in [3, 15] and len(both) >= 3:
            s_mod = s.copy()
            R_d = s_mod[:, 63:].reshape(-1, 21, 3)
            for t in both:
                idx_deltas = R_d[t, 5:9] - R_d[t, 5:6]
                R_d[t, 9:13] = R_d[t, 9:10] + idx_deltas * 1.05
                R_d[t, 13:17] = R_d[t, 13:14] + idx_deltas * 0.95
                R_d[t, 17:21] = R_d[t, 17:18] + idx_deltas * 0.80
            s_mod[:, 63:] = R_d.reshape(-1, 63)
            base_seqs.append(clean_and_normalize_seq(s_mod))
            base_labels.append(y)
        if y == 1 and len(both) >= 3:
            s_rot = s.copy()
            R_b = s_rot[:, 63:].reshape(-1, 21, 3)
            for t in both:
                R_b[t] = rotate_hand_2d(R_b[t], 18.0)
                scale_b = np.linalg.norm(R_b[t, :, :2] - R_b[t, 0:1, :2], axis=1).max()
                R_b[t, 9:13, 1] += 0.15 * scale_b
                R_b[t, 13:17, 1] += 0.30 * scale_b
            s_rot[:, 63:] = R_b.reshape(-1, 63)
            base_seqs.append(clean_and_normalize_seq(s_rot))
            base_labels.append(y)
        if idx_s == 49:
            s_occ = np.zeros_like(s)
            L_y = L_seq.copy()
            for t in range(len(s)):
                if np.any(L_y[t] != 0):
                    L_y[t, 9:21] = L_y[t, 0:1] + (L_y[t, 9:21] - L_y[t, 0:1]) * 0.45
            s_occ[:, 63:] = L_y.reshape(-1, 63)
            base_seqs.append(clean_and_normalize_seq(s_occ))
            base_labels.append(y)
        if idx_s == 46 and len(X_raw) > 22:
            s_x1 = X_raw[22].copy()
            R_x = s_x1[:, 63:].reshape(-1, 21, 3)
            for t in range(len(s_x1)):
                if np.any(R_x[t] != 0):
                    h = R_x[t]
                    scale_x = np.linalg.norm(h[:, :2] - h[0:1, :2], axis=1).max()
                    h[2, :2] = h[0, :2] + np.array([0.22, -0.22]) * scale_x
                    h[3, :2] = h[0, :2] + np.array([0.25, -0.35]) * scale_x
                    h[4, :2] = h[0, :2] + np.array([0.25, -0.47]) * scale_x
                    R_x[t] = h
            s_x1[:, 63:] = R_x.reshape(-1, 63)
            base_seqs.append(clean_and_normalize_seq(s_x1))
            base_labels.append(23)
    if os.path.exists('X_testing.npy') and os.path.exists('y_testing.npy'):
        X_te_raw = np.load('X_testing.npy')
        y_te_raw = np.load('y_testing.npy')
        for s_te, y_te in zip(X_te_raw, y_te_raw):
            base_seqs.append(clean_and_normalize_seq(s_te))
            base_labels.append(y_te)
    ref_means = np.mean(np.array(base_seqs, dtype=np.float32), axis=1)
    return ref_means, np.array(base_labels, dtype=np.int64)

def compute_hybrid_probs(model, raw_seq, ref_means=None, ref_labels=None, num_classes=26):
    norm_seq = clean_and_normalize_seq(raw_seq, target_len=30)
    norm_mir = clean_and_normalize_seq(mirror_raw_seq(raw_seq), target_len=30)
    batch_in = np.stack([norm_seq, norm_mir], axis=0)
    preds = model.predict(batch_in, verbose=0)
    lstm_probs = np.maximum(preds[0], preds[1] * 0.88)

    if ref_means is not None and ref_labels is not None:
        seq_mean = np.mean(norm_seq, axis=0)
        mir_mean = np.mean(norm_mir, axis=0)
        dists_norm = np.linalg.norm(ref_means - seq_mean[None, :], axis=1)
        dists_mir = np.linalg.norm(ref_means - mir_mean[None, :], axis=1) * 1.08
        dists = np.minimum(dists_norm, dists_mir)
        tmpl_scores = np.zeros(num_classes, dtype=np.float32)
        for c in range(num_classes):
            c_mask = (ref_labels == c)
            if np.any(c_mask):
                min_d = np.min(dists[c_mask])
                tmpl_scores[c] = np.exp(-min_d / 1.15)
        if np.sum(tmpl_scores) > 1e-6:
            tmpl_probs = tmpl_scores / np.sum(tmpl_scores)
            combined = 0.60 * lstm_probs + 0.40 * tmpl_probs
            return combined / np.sum(combined)
    return lstm_probs

# ==============================================================================
# 2. MUAT MODEL LSTM, PEMETAAN LABEL, DAN TEMPLATE REFERENSI
# ==============================================================================
MODEL_KERAS_PATH = 'model_lstm_bisindo.h5'
if not os.path.exists(MODEL_KERAS_PATH):
    print(f"Error: {MODEL_KERAS_PATH} tidak ditemukan. Simpan model terlebih dahulu.")
    exit()

print("Memuat model LSTM...")
model = load_model(MODEL_KERAS_PATH)
ref_means, ref_labels = build_reference_templates()

label_map_rev = {}
if os.path.exists('label_map.txt'):
    with open('label_map.txt', 'r') as f:
        for line in f:
            parts = line.strip().split(':')
            if len(parts) == 2:
                cls_name, idx = parts[0], int(parts[1])
                label_map_rev[idx] = cls_name
else:
    label_map_rev = {i: chr(65 + i) for i in range(26)}

# ==============================================================================
# 3. INISIALISASI MEDIAPIPE TASKS API
# ==============================================================================
MODEL_PATH = 'hand_landmarker.task'
base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
options = vision.HandLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_hands=2,
    min_hand_detection_confidence=0.25,
    min_hand_presence_confidence=0.25
)
detector = vision.HandLandmarker.create_from_options(options)

HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),         # Ibu jari
    (0, 5), (5, 6), (6, 7), (7, 8),         # Telunjuk
    (5, 9), (9, 10), (10, 11), (11, 12),    # Tengah
    (9, 13), (13, 14), (14, 15), (15, 16),  # Manis
    (13, 17), (17, 18), (18, 19), (19, 20), # Kelingking
    (0, 17)                                 # Pangkal ke kelingking
]

# ==============================================================================
# 4. PENGATURAN BUFFER RESPONSIF DAN KAMERA REAL-TIME
# ==============================================================================
sequence_buffer = []
MAX_BUFFER_LEN = 20       # Window aktif 20 frame agar perpindahan huruf cepat
MIN_ACTIVE_FRAMES = 5     # Mulai prediksi begitu 5 frame aktif terkumpul
empty_frame_count = 0
smoothed_probs = None

cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

WINDOW_NAME = 'Pengujian Real-Time BISINDO'
cv2.namedWindow(WINDOW_NAME, cv2.WINDOW_NORMAL | cv2.WINDOW_KEEPRATIO)
window_sized = False
print("Kamera terbuka. Tekan 'q' pada jendela video untuk keluar.")

predicted_label = "Menunggu Gerakan..."
confidence = 0.0
top3_text = ""

while cap.isOpened():
    ret, raw_frame = cap.read()
    if not ret:
        print("Gagal membaca kamera.")
        break

    # PENTING: Deteksi dilakukan pada frame ASLI (tanpa flip) agar orientasi X
    # sama persis dengan video dataset pelatihan, sedangkan tampilan layar di-flip
    # agar tetap nyaman seperti cermin bagi pengguna.
    rgb_frame = cv2.cvtColor(raw_frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
    results = detector.detect(mp_image)

    display_frame = cv2.flip(raw_frame, 1)
    h, w, _ = display_frame.shape

    left_hand = np.zeros(63, dtype=np.float32)
    right_hand = np.zeros(63, dtype=np.float32)
    hand_detected = False

    if results.hand_landmarks and len(results.hand_landmarks) > 0:
        hand_detected = True
        # Urutkan tangan berdasarkan koordinat X pada frame asli (kiri ke kanan gambar asli)
        sorted_hands = sorted(results.hand_landmarks, key=lambda hl: hl[0].x)

        for i, hand_landmarks in enumerate(sorted_hands):
            # Gambar koneksi tulang jari pada display_frame (koordinat X dicerminkan: 1.0 - x)
            for connection in HAND_CONNECTIONS:
                pt1 = hand_landmarks[connection[0]]
                pt2 = hand_landmarks[connection[1]]
                x1, y1 = int((1.0 - pt1.x) * w), int(pt1.y * h)
                x2, y2 = int((1.0 - pt2.x) * w), int(pt2.y * h)
                cv2.line(display_frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

            landmarks = []
            for lm in hand_landmarks:
                x_draw, y_draw = int((1.0 - lm.x) * w), int(lm.y * h)
                cv2.circle(display_frame, (x_draw, y_draw), 5, (0, 0, 255), -1)
                landmarks.extend([lm.x, lm.y, lm.z])

            landmarks_array = np.array(landmarks, dtype=np.float32)

            # Penempatan slot tangan yang konsisten dengan dataset pelatihan:
            # - Pada video tanpa flip, tangan kanan pelaku berada di sisi kiri gambar (x lebih kecil -> right_hand [63:])
            #   dan tangan kiri pelaku berada di sisi kanan gambar (x lebih besar -> left_hand [:63]).
            # - Untuk isyarat 1 tangan, seluruh data pelatihan berada di slot right_hand ([63:]).
            if len(sorted_hands) == 2:
                if i == 0:
                    right_hand = landmarks_array
                else:
                    left_hand = landmarks_array
            else:
                right_hand = landmarks_array

    if hand_detected:
        empty_frame_count = 0
        frame_features = np.concatenate([left_hand, right_hand])
        sequence_buffer.append(frame_features)
        if len(sequence_buffer) > MAX_BUFFER_LEN:
            sequence_buffer.pop(0)

        if len(sequence_buffer) >= MIN_ACTIVE_FRAMES:
            probs = compute_hybrid_probs(model, sequence_buffer, ref_means, ref_labels)
            if smoothed_probs is None:
                smoothed_probs = probs
            else:
                smoothed_probs = 0.55 * probs + 0.45 * smoothed_probs

            max_idx = int(np.argmax(smoothed_probs))
            confidence = float(smoothed_probs[max_idx])
            top3_idx = np.argsort(smoothed_probs)[::-1][:3]
            top3_text = " | ".join([f"{label_map_rev.get(int(k), '?')}:{smoothed_probs[k]*100:.0f}%" for k in top3_idx])

            if confidence >= 0.20:
                predicted_label = label_map_rev.get(max_idx, f"Kelas {max_idx}")
            else:
                predicted_label = "Tidak Terdeteksi"
    else:
        empty_frame_count += 1
        if empty_frame_count >= 5:
            sequence_buffer.clear()
            smoothed_probs = None
            predicted_label = "Menunggu Gerakan..."
            confidence = 0.0
            top3_text = ""

    # Tampilan antarmuka visual
    cv2.rectangle(display_frame, (0, 0), (w, 75), (245, 117, 16), -1)
    text_display = f"Prediksi BISINDO: {predicted_label} ({confidence*100:.1f}%)"
    cv2.putText(display_frame, text_display, (10, 32), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
    if top3_text:
        cv2.putText(display_frame, f"Top-3: {top3_text}", (10, 62), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (230, 255, 230), 1, cv2.LINE_AA)

    if not window_sized:
        target_w = 960
        target_h = max(1, int(target_w * h / w))
        cv2.resizeWindow(WINDOW_NAME, target_w, target_h)
        window_sized = True

    cv2.imshow(WINDOW_NAME, display_frame)

    if cv2.waitKey(10) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Memuat model LSTM...


Kamera terbuka. Tekan 'q' pada jendela video untuk keluar.
